## Parte 1 — Softmax con temperatura

La softmax con temperatura se implementa de forma numéricamente estable restando el máximo de los logits antes de exponenciar: p_i = exp((z_i − max z)/T) / Σ_j exp((z_j − max z)/T). Esto evita desbordamientos sin alterar el resultado, pues el factor común exp(max z/T) se cancela entre numerador y denominador. Sobre los logits [2.0, 1.0, 0.5, 0.2, −1.0, −3.0] se obtienen las siguientes distribuciones (cada fila suma 1.0) y entropías en bits:

| Temperatura | t0 | t1 | t2 | t3 | t4 | t5 | Entropía (bits) |
|---|---|---|---|---|---|---|---|
| 0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 |
| 1 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 |
| 2 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 |

El efecto de la temperatura es claro: al bajarla de 1 a 0.5, la masa se concentra en el token de mayor logit (t0 pasa de 0.5516 a 0.8231) y la entropía cae de 1.7998 a 0.9147 bits; t5 queda con una probabilidad de 3.7367e-05, que se redondea a 0.0000 en la tabla. Al subir la temperatura a 2, la distribución se aplana (t0 baja a 0.3583 y t5 sube a 0.0294) y la entropía sube a 2.2888 bits.

In [1]:
# T1: código aprobado por el crítico
# T1 — Softmax con temperatura (NumPy, estable) + entropía en bits
# Tarea B · MMIA 6013 · Taller 03 v2

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ datos
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
TEMPERATURAS = [0.5, 1.0, 2.0]


# --------------------------------------------- softmax con temperatura
def softmax_con_temperatura(logits: np.ndarray, T: float) -> np.ndarray:
    """p_i = exp(z_i/T) / sum_j exp(z_j/T), numéricamente estable
    (se resta el máximo de los logits escalados antes de la exponencial)."""
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - np.max(z)  # estabilidad numérica
    e = np.exp(z)
    return e / e.sum()


def entropia_bits(p: np.ndarray) -> float:
    """Entropía de Shannon H(p) = -sum p_i log2(p_i), en bits."""
    p = np.asarray(p, dtype=float)
    p_nz = p[p > 0]
    return float(-np.sum(p_nz * np.log2(p_nz)))


# ------------------------------------------------------------- cálculos
distribuciones = {T: softmax_con_temperatura(LOGITS, T) for T in TEMPERATURAS}
entropias = {T: entropia_bits(p) for T, p in distribuciones.items()}
sumas = {T: float(p.sum()) for T, p in distribuciones.items()}

# Verificación del criterio de éxito: cada fila de probabilidades suma 1
for T, s in sumas.items():
    assert abs(s - 1.0) < 1e-12, f"La distribución T={T} no suma 1 (suma={s})"

# ------------------------------------------- tabla con cuatro decimales
filas = []
for T in TEMPERATURAS:
    fila = {"Temperatura": f"{T:g}"}
    for tok, pi in zip(TOKENS, distribuciones[T]):
        fila[tok] = f"{pi:.4f}"
    fila["Entropia_bits"] = f"{entropias[T]:.4f}"
    filas.append(fila)

tabla = pd.DataFrame(filas)

# Tabla en formato markdown (sin dependencias externas)
claves = ["Temperatura"] + TOKENS + ["Entropia_bits"]
lineas = [
    "| " + " | ".join(claves) + " |",
    "|" + "---|" * len(claves),
]
for fila in filas:
    lineas.append("| " + " | ".join(str(fila[k]) for k in claves) + " |")
tabla_md = "\n".join(lineas)

# ------------------------------------------------------- resultados.json
resultados = {
    "tokens": TOKENS,
    "logits": LOGITS.tolist(),
    "temperaturas": TEMPERATURAS,
    "softmax_numerically_estable": True,  # se resta el máximo de los logits
    "distribuciones": {f"T={T:g}": distribuciones[T].tolist() for T in TEMPERATURAS},
    "entropias_bits": {f"T={T:g}": entropias[T] for T in TEMPERATURAS},
    "suma_de_cada_fila": {f"T={T:g}": sumas[T] for T in TEMPERATURAS},
    "tabla_4_decimales": filas,
    "tabla_markdown_4_decimales": tabla_md,
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------- figura
fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(TOKENS))
ancho = 0.25
colores = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for i, T in enumerate(TEMPERATURAS):
    ax.bar(x + (i - 1) * ancho, distribuciones[T], width=ancho,
           color=colores[i], label=f"T={T:g}  (H = {entropias[T]:.4f} bits)")
ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad")
ax.set_title("Softmax con temperatura — logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------- resumen
print("T1 — Softmax con temperatura (implementación NumPy estable, restando el máximo)")
print("Logits:", LOGITS.tolist())
print()
print(tabla.to_string(index=False))
print()
for T in TEMPERATURAS:
    print(f"T={T:g}: suma de probabilidades = {sumas[T]:.12f} | "
          f"entropía = {entropias[T]:.4f} bits")
print()
print("Archivos generados: resultados.json, t1_softmax_temperatura.png")


T1 — Softmax con temperatura (implementación NumPy estable, restando el máximo)
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]

Temperatura     t0     t1     t2     t3     t4     t5 Entropia_bits
        0.5 0.8231 0.1114 0.0410 0.0225 0.0020 0.0000        0.9147
          1 0.5516 0.2029 0.1231 0.0912 0.0275 0.0037        1.7998
          2 0.3583 0.2173 0.1693 0.1457 0.0800 0.0294        2.2888

T=0.5: suma de probabilidades = 1.000000000000 | entropía = 0.9147 bits
T=1: suma de probabilidades = 1.000000000000 | entropía = 1.7998 bits
T=2: suma de probabilidades = 1.000000000000 | entropía = 2.2888 bits

Archivos generados: resultados.json, t1_softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1 se aplica top-p con p = 0.9: se ordenan los tokens por probabilidad descendente (el orden resulta ser t0, t1, t2, t3, t4, t5) y se conserva el conjunto mínimo cuya acumulada alcanza 0.9. Con t0, t1 y t2 la acumulada es 0.8776, aún por debajo de 0.9, por lo que es necesario incluir t3; en ese punto la acumulada del corte es 0.9688 ≥ 0.9. Sobreviven t0, t1, t2 y t3; quedan eliminados t4 y t5, y las probabilidades de los supervivientes se renormalizan (la suma da 1.0 con error 0.0):

| Posición | Token | p (T=1) | Acumulada | ¿Sobrevive? | p renormalizada |
|---|---|---|---|---|---|
| 1 | t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| 2 | t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| 3 | t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| 4 | t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| 5 | t4 | 0.0275 | 0.9963 | no | 0.0000 |
| 6 | t5 | 0.0037 | 1.0000 | no | 0.0000 |

La distribución renormalizada es t0 = 0.5694, t1 = 0.2095, t2 = 0.1270, t3 = 0.0941 (t4 = t5 = 0.0). El recorte reduce la entropía de 1.7998 bits (distribución original de T = 1) a 1.6341 bits, lo que cuantifica la pérdida de incertidumbre al eliminar la cola de baja probabilidad.

In [2]:
# T2: código aprobado por el crítico
# T2: Muestreo de núcleo (top-p) con p=0.9 sobre la distribución de T=1 de la Parte 1
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Parámetros fijados por el enunciado
# ------------------------------------------------------------------
P = 0.9
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)


def softmax_con_temperatura(logits, T):
    """Softmax con temperatura, numéricamente estable (resta del máximo)."""
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - np.max(z)
    e = np.exp(z)
    return e / np.sum(e)


# ------------------------------------------------------------------
# 1) Distribución de T=1 obtenida en la Parte 1 (entrada/T1/resultados.json)
# ------------------------------------------------------------------
ruta_T1 = Path("entrada") / "T1" / "resultados.json"
dist_T1 = None
fuente = None
if ruta_T1.exists():
    with ruta_T1.open("r", encoding="utf-8") as f:
        res_T1 = json.load(f)
    d = res_T1.get("distribuciones", {}).get("T=1")
    if d is not None:
        dist_T1 = np.asarray(d, dtype=float)
        fuente = "entrada/T1/resultados.json (distribucion de T=1 de la Parte 1)"
if dist_T1 is None:
    # Respaldo: recalcular exactamente como en la Parte 1
    dist_T1 = softmax_con_temperatura(LOGITS, 1.0)
    fuente = "recalculada con softmax numericamente estable (T=1)"

# Verificación de procedencia: coincide con la softmax estable de la Parte 1
dist_T1_recalc = softmax_con_temperatura(LOGITS, 1.0)
max_abs_diff = float(np.max(np.abs(dist_T1 - dist_T1_recalc)))

# ------------------------------------------------------------------
# 2) Top-p (nucleus) con p = 0.9
#    Ordenar por probabilidad descendente, conservar el conjunto MINIMO
#    cuya acumulada alcanza p, y renormalizar los supervivientes.
# ------------------------------------------------------------------
orden = np.argsort(-dist_T1, kind="stable")          # índices en orden descendente
probs_ord = dist_T1[orden]
acum_ord = np.cumsum(probs_ord)

# primer índice (en el orden) cuya acumulada >= p  ->  conjunto mínimo
k = int(np.searchsorted(acum_ord, P, side="left"))
k = min(k, len(TOKENS) - 1)
supervivientes_ord = orden[: k + 1]
acum_corte = float(acum_ord[k])

es_sup = np.zeros(len(TOKENS), dtype=bool)
es_sup[supervivientes_ord] = True

dist_renorm = np.zeros(len(TOKENS), dtype=float)
dist_renorm[es_sup] = dist_T1[es_sup] / acum_corte   # renormalización

tokens_sup = [TOKENS[i] for i in supervivientes_ord]
tokens_eliminados = [TOKENS[i] for i in range(len(TOKENS)) if not es_sup[i]]
probs_sup = [float(dist_T1[i]) for i in supervivientes_ord]
acum_por_sup = {TOKENS[orden[j]]: float(acum_ord[j]) for j in range(k + 1)}
dist_renorm_dict = {TOKENS[i]: float(dist_renorm[i]) for i in supervivientes_ord}

suma_renorm = float(dist_renorm[es_sup].sum())
p_ren = dist_renorm[es_sup]
entropia_ren_bits = float(-np.sum(p_ren * np.log2(p_ren)))
entropia_T1_bits = float(-np.sum(dist_T1 * np.log2(dist_T1)))

# ------------------------------------------------------------------
# 3) Tabla del proceso (precisión completa + markdown a 4 decimales)
# ------------------------------------------------------------------
tabla = []
for j, idx in enumerate(orden):
    tabla.append({
        "posicion": j + 1,
        "token": TOKENS[idx],
        "prob_T1": float(dist_T1[idx]),
        "acumulada": float(acum_ord[j]),
        "sobrevive": bool(es_sup[idx]),
        "prob_renormalizada": float(dist_renorm[idx]),
    })

filas_md = []
for fila in tabla:
    filas_md.append(
        f"| {fila['posicion']} | {fila['token']} | {fila['prob_T1']:.4f} | "
        f"{fila['acumulada']:.4f} | {'si' if fila['sobrevive'] else 'no'} | "
        f"{fila['prob_renormalizada']:.4f} |"
    )
tabla_md = (
    "| Posicion | Token | p (T=1) | Acumulada | Sobrevive? | p renormalizada |\n"
    "|---|---|---|---|---|---|\n" + "\n".join(filas_md)
)

# ------------------------------------------------------------------
# 4) Figura: distribución original vs renormalizada + curva acumulada
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
x = np.arange(len(TOKENS))
ax.bar(x - 0.2, dist_T1, width=0.4, color="#8d99ae", label="Distribucion T=1 (Parte 1)")
ax.bar(x + 0.2, dist_renorm, width=0.4,
       color=["#2a9d8f" if s else "#e63946" for s in es_sup],
       label="Top-p renormalizada")
ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_ylabel("Probabilidad")
ax.set_title(f"Top-p (p={P}) sobre la distribucion de T=1")
ax.legend(fontsize=8)

ax = axes[1]
ax.step(np.arange(1, len(TOKENS) + 1), acum_ord, where="post",
        marker="o", color="#264653", label="Acumulada (orden descendente)")
ax.axhline(P, color="#e76f51", ls="--", label=f"p = {P}")
ax.axvline(k + 1, color="#2a9d8f", ls=":", label="Corte del nucleo")
ax.set_xticks(np.arange(1, len(TOKENS) + 1))
ax.set_xticklabels([TOKENS[i] for i in orden])
ax.set_ylim(0, 1.05)
ax.set_xlabel("Tokens ordenados por probabilidad descendente")
ax.set_ylabel("Probabilidad acumulada")
ax.set_title("Probabilidad acumulada y corte top-p")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("T2_top_p_nucleus.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# 5) resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "subtarea": "T2_top_p_nucleus",
    "p": P,
    "temperatura": 1.0,
    "fuente_distribucion_T1": fuente,
    "tokens": TOKENS,
    "distribucion_T1_parte1": [float(v) for v in dist_T1],
    "verificacion_coincidencia_softmax_T1_max_abs_diff": max_abs_diff,
    "orden_descendente": [TOKENS[i] for i in orden],
    "probs_ordenadas": [float(v) for v in probs_ord],
    "acumulada_ordenada": [float(v) for v in acum_ord],
    "tokens_supervivientes": tokens_sup,
    "probabilidades_originales_supervivientes": probs_sup,
    "probabilidad_acumulada_por_token_superviviente": acum_por_sup,
    "probabilidad_acumulada_del_corte": acum_corte,
    "probabilidad_acumulada_alcanza_p": bool(acum_corte >= P),
    "tokens_eliminados": tokens_eliminados,
    "distribucion_renormalizada": dist_renorm_dict,
    "distribucion_renormalizada_vector_alineado_t0_t5": [float(v) for v in dist_renorm],
    "suma_distribucion_renormalizada": suma_renorm,
    "error_suma_vs_1": abs(suma_renorm - 1.0),
    "entropia_bits_T1_original": entropia_T1_bits,
    "entropia_bits_renormalizada": entropia_ren_bits,
    "tabla_proceso": tabla,
    "tabla_markdown_4_decimales": tabla_md,
    "figura": "T2_top_p_nucleus.png",
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------
# 6) Resumen
# ------------------------------------------------------------------
print("=== T2: Muestreo de nucleo (top-p) con p=0.9 sobre la distribucion de T=1 ===")
print(f"Fuente de la distribucion T=1: {fuente}")
print(f"Verificacion vs softmax estable de la Parte 1 (max |diff|): {max_abs_diff:.3e}")
print(f"Orden descendente: {[TOKENS[i] for i in orden]}")
print(f"Acumulada por token: {acum_por_sup}")
print(f"Tokens supervivientes: {tokens_sup} (acumulada del corte = {acum_corte:.10f} >= {P})")
print(f"Tokens eliminados: {tokens_eliminados}")
print("Distribucion renormalizada:")
for tok, val in dist_renorm_dict.items():
    print(f"  {tok}: {val:.10f}")
print(f"Suma de la distribucion renormalizada: {suma_renorm:.12f}")
print(f"Entropia (bits): T=1 original = {entropia_T1_bits:.6f} | renormalizada = {entropia_ren_bits:.6f}")
print("Figura guardada: T2_top_p_nucleus.png | Resultados en resultados.json")


=== T2: Muestreo de nucleo (top-p) con p=0.9 sobre la distribucion de T=1 ===
Fuente de la distribucion T=1: entrada/T1/resultados.json (distribucion de T=1 de la Parte 1)
Verificacion vs softmax estable de la Parte 1 (max |diff|): 0.000e+00
Orden descendente: ['t0', 't1', 't2', 't3', 't4', 't5']
Acumulada por token: {'t0': 0.5516226378718239, 't1': 0.7545532656296273, 't2': 0.8776369131594667, 't3': 0.968819521917538}
Tokens supervivientes: ['t0', 't1', 't2', 't3'] (acumulada del corte = 0.9688195219 >= 0.9)
Tokens eliminados: ['t4', 't5']
Distribucion renormalizada:
  t0: 0.5693760555
  t1: 0.2094617451
  t2: 0.1270449704
  t3: 0.0941172290
Suma de la distribucion renormalizada: 1.000000000000
Entropia (bits): T=1 original = 1.799836 | renormalizada = 1.634075
Figura guardada: T2_top_p_nucleus.png | Resultados en resultados.json


## Parte 3 — Comprobación empírica

Se tomaron 10000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Los conteos observados fueron 5682 para t0, 2093 para t1, 1296 para t2, 929 para t3 y 0 para t4 y t5 (estos dos últimos tienen probabilidad teórica nula tras el recorte). La comparación con las probabilidades teóricas es:

| Token | p teórica | Conteo observado | Frecuencia observada | Diferencia (obs − teo) |
|---|---|---|---|---|
| t0 | 0.5694 | 5682 | 0.5682 | −0.0012 |
| t1 | 0.2095 | 2093 | 0.2093 | −0.0002 |
| t2 | 0.1270 | 1296 | 0.1296 | 0.0026 |
| t3 | 0.0941 | 929 | 0.0929 | −0.0012 |
| t4 | 0.0000 | 0 | 0.0000 | 0.0000 |
| t5 | 0.0000 | 0 | 0.0000 | 0.0000 |

Las frecuencias observadas reproducen las teóricas con desviaciones del orden de 10⁻³, coherentes con el ruido de muestreo a n = 10000. La divergencia KL de la distribución empírica respecto de la teórica es de 5.0069e-05 bits, un valor muy pequeño y del mismo orden que la referencia asintótica esperada para este tamaño de muestra (2.1640e-04 bits), lo que confirma que el generador muestrea la distribución renormalizada correctamente. El gráfico de barras de frecuencias observadas frente a teóricas muestra las parejas de barras prácticamente indistinguibles.

In [3]:
# T3: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Subtarea T3 — Parte 3: Comprobación empírica del muestreo top-p
#   * 10,000 muestras de la distribución renormalizada de la Parte 2
#   * generador numpy.random.default_rng(0)
#   * Diagrama de barras: frecuencias observadas vs. probabilidades teóricas
#   * Divergencia KL (en bits) de la distribución empírica respecto de la teórica
# ----------------------------------------------------------------------

N_MUESTRAS = 10_000
SEMILLA = 0
P_TOP = 0.9
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
FIGURA = "T3_frecuencias_vs_teorica.png"


def softmax_T1(logits):
    """Softmax con T=1, numéricamente estable (resta del máximo)."""
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()


def top_p_renormalizada(p, p_top):
    """Conjunto mínimo de tokens cuya acumulada alcanza p_top; renormaliza."""
    orden = np.argsort(-p, kind="stable")
    acum = np.cumsum(p[orden])
    k = int(np.searchsorted(acum, p_top, side="left")) + 1
    sup = np.sort(orden[:k])
    q = np.zeros_like(p)
    q[sup] = p[sup] / p[sup].sum()
    return q, [TOKENS[i] for i in sup]


# --- 1) Distribución renormalizada de la Parte 2 (salida de T2) --------
ruta_t2 = Path("entrada") / "T2" / "resultados.json"
fuente = "entrada/T2/resultados.json"
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        t2 = json.load(f)
    p_renorm = np.asarray(t2["distribucion_renormalizada_vector_alineado_t0_t5"], dtype=float)
    tokens_sup = list(t2["tokens_supervivientes"])
else:
    fuente = "recalculada en T3 (softmax T=1 + top-p 0.9)"
    p_renorm, tokens_sup = top_p_renormalizada(softmax_T1(LOGITS), P_TOP)

# Verificación determinista: el recálculo debe coincidir con T2
p_renorm_check, tokens_sup_check = top_p_renormalizada(softmax_T1(LOGITS), P_TOP)
max_diff_check = float(np.max(np.abs(p_renorm_check - p_renorm)))

# --- 2) Muestreo: 10,000 muestras con default_rng(0) -------------------
p = p_renorm / p_renorm.sum()  # garantiza suma exactamente 1 para rng.choice
rng = np.random.default_rng(SEMILLA)
muestras = rng.choice(len(TOKENS), size=N_MUESTRAS, p=p)
conteos = np.bincount(muestras, minlength=len(TOKENS))
frecuencias = conteos / N_MUESTRAS

# --- 3) Divergencia KL en bits: D(empírica || teórica) -----------------
mask = frecuencias > 0  # los términos con q_i = 0 aportan 0 a la suma
kl_bits = float(np.sum(frecuencias[mask] * np.log2(frecuencias[mask] / p[mask])))
kl_nats = float(np.sum(frecuencias[mask] * np.log(frecuencias[mask] / p[mask])))
# Referencia asintótica: E[KL] ≈ (k-1)/(2n) nats con k categorías con p>0
kl_ref_bits = float((len(tokens_sup) - 1) / (2.0 * N_MUESTRAS * np.log(2.0)))

# --- 4) Diagrama de barras: observado vs. teórico ----------------------
idx_sup = [TOKENS.index(t) for t in tokens_sup]
probs_sup = p_renorm[idx_sup]
freqs_sup = frecuencias[idx_sup]

x = np.arange(len(tokens_sup))
w = 0.38
fig, ax = plt.subplots(figsize=(8.0, 5.0))
b_teo = ax.bar(x - w / 2, probs_sup, w, color="#4C72B0", edgecolor="black",
               linewidth=0.5, label="Probabilidad teórica (renormalizada)")
b_obs = ax.bar(x + w / 2, freqs_sup, w, color="#DD8452", edgecolor="black",
               linewidth=0.5, label=f"Frecuencia observada (n={N_MUESTRAS:,})")
for barras in (b_teo, b_obs):
    for rect in barras:
        h = rect.get_height()
        ax.annotate(f"{h:.4f}",
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 2), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(tokens_sup)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / Frecuencia")
ax.set_ylim(0.0, 1.15 * max(probs_sup.max(), freqs_sup.max()))
ax.set_title("Parte 3 — Muestreo top-p (T=1, p=0.9): observado vs. teórico\n"
             f"10,000 muestras, numpy.random.default_rng({SEMILLA}) · KL = {kl_bits:.6f} bits")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURA, dpi=120)
plt.close(fig)

# --- 5) resultados.json (contrato de la subtarea) ----------------------
resultados = {
    "subtarea": "T3_comprobacion_empirica",
    "parte": "Parte 3 — Comprobación empírica",
    "fuente_distribucion_renormalizada": fuente,
    "temperatura": 1.0,
    "p_top": P_TOP,
    "semilla": SEMILLA,
    "generador": f"numpy.random.default_rng({SEMILLA})",
    "n_muestras": N_MUESTRAS,
    "tokens": TOKENS,
    "tokens_supervivientes": tokens_sup,
    "probabilidades_teoricas_renormalizadas": {
        t: float(p_renorm[i]) for i, t in enumerate(TOKENS) if p_renorm[i] > 0.0
    },
    "probabilidades_teoricas_alineadas_t0_t5": [float(v) for v in p_renorm],
    "conteos_observados": {t: int(conteos[i]) for i, t in enumerate(TOKENS)},
    "frecuencias_observadas": {t: float(frecuencias[i]) for i, t in enumerate(TOKENS)},
    "frecuencias_observadas_alineadas_t0_t5": [float(v) for v in frecuencias],
    "suma_frecuencias_observadas": float(frecuencias.sum()),
    "divergencia_KL_bits": kl_bits,
    "divergencia_KL_nats": kl_nats,
    "KL_esperada_bits_referencia_asintotica": kl_ref_bits,
    "verificacion_recalculo_softmax_topp_max_abs_diff": max_diff_check,
    "tokens_supervivientes_verificacion": tokens_sup_check,
    "figura": FIGURA,
    "tabla_comparativa": [
        {
            "token": tok,
            "prob_teorica_renormalizada": float(p_renorm[TOKENS.index(tok)]),
            "conteo_observado": int(conteos[TOKENS.index(tok)]),
            "frecuencia_observada": float(frecuencias[TOKENS.index(tok)]),
            "diferencia_obs_menos_teo": float(
                frecuencias[TOKENS.index(tok)] - p_renorm[TOKENS.index(tok)]
            ),
        }
        for tok in tokens_sup
    ],
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# --- 6) Resumen ---------------------------------------------------------
print("=== T3 — Comprobación empírica (Parte 3) ===")
print(f"Fuente distribución renormalizada   : {fuente}")
print(f"Tokens supervivientes (top-p {P_TOP})    : {tokens_sup}")
print(f"Muestras: {N_MUESTRAS} con numpy.random.default_rng({SEMILLA})")
for tok in tokens_sup:
    i = TOKENS.index(tok)
    print(f"  {tok}: teórica={p_renorm[i]:.6f}  observada={frecuencias[i]:.6f}  (conteo={conteos[i]})")
print(f"Divergencia KL(empírica || teórica) = {kl_bits:.6f} bits ({kl_nats:.6f} nats)")
print(f"Referencia asintótica E[KL] ≈ {kl_ref_bits:.6e} bits")
print(f"Verificación recálculo softmax+top-p: max|diff| = {max_diff_check:.3e}")
print(f"Figura guardada: {FIGURA}")
print("Resultados escritos en resultados.json")


=== T3 — Comprobación empírica (Parte 3) ===
Fuente distribución renormalizada   : entrada/T2/resultados.json
Tokens supervivientes (top-p 0.9)    : ['t0', 't1', 't2', 't3']
Muestras: 10000 con numpy.random.default_rng(0)
  t0: teórica=0.569376  observada=0.568200  (conteo=5682)
  t1: teórica=0.209462  observada=0.209300  (conteo=2093)
  t2: teórica=0.127045  observada=0.129600  (conteo=1296)
  t3: teórica=0.094117  observada=0.092900  (conteo=929)
Divergencia KL(empírica || teórica) = 0.000050 bits (0.000035 nats)
Referencia asintótica E[KL] ≈ 2.164043e-04 bits
Verificación recálculo softmax+top-p: max|diff| = 0.000e+00
Figura guardada: T3_frecuencias_vs_teorica.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

Cuando T → 0, los exponentes (z_i − max z)/T divergen: el token de mayor logit (t0, con z = 2.0) mantiene exponente 0, mientras que todos los demás tienden a −∞, de modo que toda la masa se concentra en t0 y la distribución degenera en una delta; su entropía tiende a 0 bits. Cuando T → ∞, en cambio, todos los exponentes z_i/T tienden a 0, los términos exp(z_i/T) se igualan y la distribución tiende a la uniforme sobre los seis tokens, que es la de entropía máxima posible para ese vocabulario.

Las entropías medidas en la Parte 1 evidencian ambos extremos como tendencia: al bajar la temperatura la entropía cae (2.2888 bits en T = 2 → 1.7998 bits en T = 1 → 0.9147 bits en T = 0.5) y la masa se concentra en t0 (0.3583 → 0.5516 → 0.8231); al subirla, las probabilidades se igualan (en T = 2 la brecha entre t0 y t5 ya se redujo a 0.3583 frente a 0.0294, cuando en T = 0.5 era de 0.8231 frente a 0.0000).

T → 0 equivale a la decodificación voraz porque, para cualquier temperatura fija, la softmax es una transformación monótona creciente de los logits: el argmax de la distribución coincide siempre con el argmax de los logits. La decodificación voraz elige en cada paso el token de mayor logit, es decir, t0; cuando T → 0 la distribución muestreada pone toda su masa precisamente en ese token, de modo que muestrear de ella produce exactamente la misma salida que greedy. La caída de la entropía hacia 0 bits (0.9147 bits ya en T = 0.5, con t0 concentrando 0.8231 de la masa) muestra que la distribución se vuelve determinista, que es justamente la condición bajo la cual muestreo y greedy coinciden.